In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 09 · Inferencia · del modelo registrado a un precio que cualquiera puede pedir

> Especificación: `especificaciones/07-validacion-registro-app.md`, secciones 7 y 8 · Guía de la clase 4, secciones 6 a 9.

Un modelo entrenado no es un producto. Un producto recibe datos **incompletos o equivocados de una persona real** y responde
con algo útil o con un **error claro**. Esta etapa:

1. **Exporta** el modelo `champion` del registro a `models/model.joblib`, el archivo que viaja con la app.
2. Escribe el **esquema de entrada** (`models/input_schema.json`): columnas, rangos posibles, barrios de cada ciudad y el
   rango típico del precio.
3. Prueba `predict_price` con un inmueble normal, uno incompleto y uno **imposible**.
4. Comprueba que el archivo exportado **es** el modelo que se evaluó en la etapa 7 (segunda lectura de la prueba, anotada en
   la marca, que **no decide nada**).

## 0. Datos y resultados de las etapas 7 y 8

In [2]:
import json

import joblib
import numpy as np
import pandas as pd

from tarifalatam.config import (
    MODEL_ALIAS,
    PROCESSED_DIR,
    REFERENCE_BINARY,
    REFERENCE_CATEGORICAL,
    REFERENCE_NUMERIC,
    REPORTS_DIR,
    ROOT,
    VALID_RANGE,
)
from tarifalatam.data.load import load_train

COLUMNAS = REFERENCE_CATEGORICAL + REFERENCE_NUMERIC + REFERENCE_BINARY
etapa7 = json.loads((REPORTS_DIR / "07_metricas_test.json").read_text(encoding="utf-8"))
registro = json.loads((REPORTS_DIR / "08_registro.json").read_text(encoding="utf-8"))
train = load_train()
factor = etapa7["prueba"]["factor_aplicado"]
print(f"modelo registrado: {registro['modelo_registrado']} v{registro['version']} @{registro['alias']} · factor del logaritmo: {factor}")

modelo registrado: tarifalatam_precio_noche v1 @champion · factor del logaritmo: 1.0


## 1. Exportar el modelo `champion`

La fuente de verdad es el **registro**. Si no responde, el notebook se detiene: no se usa una copia local en silencio.

In [3]:
import mlflow

from tarifalatam.tracking.mlflow_utils import configure_mlflow

donde = configure_mlflow(force_local=True)
modelo = mlflow.sklearn.load_model(f"models:/{registro['modelo_registrado']}@{MODEL_ALIAS}")
destino = ROOT / "models" / "model.joblib"
joblib.dump(modelo, destino)
megas = destino.stat().st_size / 1e6
assert megas < 100, "GitHub no acepta archivos de más de 100 MB"
print(f"exportado desde el registro ({donde}) a models/model.joblib · {megas:.1f} MB")

exportado desde el registro (local) a models/model.joblib · 10.3 MB


## 2. El esquema de entrada

Se calcula con el **entrenamiento**. `VALID_RANGE` es conocimiento del grupo (lo **posible**: 1 a 16 huéspedes, 1 a 29
noches mínimas…); fuera de él la predicción **se niega**. Fuera de lo **observado** solo hay advertencia. El rango típico y
el factor salen de la etapa 7 (anfitriones reservados). El aviso de Belice y el error por ciudad (`city_mae`) **informan** el error medido en la prueba (son lo único del esquema que sale de la prueba; no cambian ninguna predicción, rango ni regla). Para el mapa, el esquema guarda la mediana del precio de cada barrio con 30 anuncios o más, calculada con el **entrenamiento**.

In [4]:
from tarifalatam.inference.schema import build_schema

por_ciudad = {c["city"]: c["mae_model"] for c in etapa7["prueba"]["por_ciudad"]}
mae_global = etapa7["prueba"]["modelo"]["mae"]
aviso_belice = (
    f"En Belice el modelo se equivoca mucho más: en la prueba, {por_ciudad['belize']:.0f} USD por noche en promedio "
    f"(frente a {mae_global:.0f} USD en las seis ciudades), y suele quedarse corto en los inmuebles caros. "
    "Úsalo solo como una referencia inicial."
)
schema = build_schema(
    train, REFERENCE_NUMERIC, REFERENCE_CATEGORICAL, REFERENCE_BINARY, VALID_RANGE,
    price_range_ratios=etapa7["decisiones"]["rango_tipico"], correction_factor=factor,
    city_warnings={"belize": aviso_belice}, city_mae=por_ciudad, path=ROOT / "models" / "input_schema.json",
)
print(f"esquema guardado: {len(COLUMNAS)} columnas · {len(schema['categories']['city'])} ciudades · "
      f"{sum(len(b) for b in schema['neighbourhoods_by_city'].values())} barrios")
print("rango posible:", schema["valid_range"])

esquema guardado: 13 columnas · 6 ciudades · 322 barrios
rango posible: {'accommodates': [1.0, 16.0], 'bedrooms': [0.0, 10.0], 'beds': [0.0, 50.0], 'bathrooms_count': [0.0, 20.0], 'minimum_nights': [1.0, 29.0], 'maximum_nights': [1.0, 1125.0], 'latitude': [-90.0, 90.0], 'longitude': [-180.0, 180.0], 'bathroom_shared': [0.0, 1.0]}


## 3. Probar la predicción

Tres casos: un inmueble normal, uno **incompleto** («No sé» en baños, tipo de propiedad y barrio) y uno **imposible**
(40 huéspedes). Qué se espera: los dos primeros responden; el tercero **se niega y explica qué corregir**.

In [5]:
from tarifalatam.inference.predict import InputValidationError, load_model, predict_price

exportado = load_model(destino)
normal = train[COLUMNAS].head(1).copy()
print("— inmueble normal —")
print(normal.T.to_string(header=False))
print(predict_price(normal, exportado, schema).to_string(index=False))

incompleto = normal.copy()
incompleto[["bathrooms_count", "property_type", "neighbourhood_cleansed"]] = np.nan
print("\n— con «No sé» en baños, tipo de propiedad y barrio —")
print(predict_price(incompleto, exportado, schema).to_string(index=False))

imposible = normal.assign(accommodates=40)
try:
    predict_price(imposible, exportado, schema)
    print("\nERROR: debía negarse a predecir con 40 huéspedes")
except InputValidationError as error:
    print("\n— con 40 huéspedes — se niega:", error)

— inmueble normal —
city                     rio-de-janeiro
neighbourhood_cleansed       Copacabana
room_type               Entire home/apt
property_type              Entire condo
latitude                      -22.96599
longitude                      -43.1794
accommodates                          5
bedrooms                            2.0
beds                                2.0
bathrooms_count                     1.0
minimum_nights                      1.0
maximum_nights                     28.0
bathroom_shared                     0.0
          city  price_usd  low_usd  high_usd
rio-de-janeiro        114       83       146

— con «No sé» en baños, tipo de propiedad y barrio —
          city  price_usd  low_usd  high_usd
rio-de-janeiro        137       99       175

— con 40 huéspedes — se niega: Huéspedes (accommodates): debe estar entre 1 y 16


## 4. ¿El archivo exportado es el modelo que se evaluó?

Segunda lectura de la prueba, **solo para comprobar**: el MAE con `models/model.joblib` debe ser idéntico al de
`reports/07_metricas_test.json`. Queda anotada en `reports/prueba_abierta.txt` con su propósito.

In [6]:
from tarifalatam.models.validation import open_test

test = open_test(PROCESSED_DIR / "test.csv.gz", purpose="comprobar que models/model.joblib es el modelo evaluado (no decide nada)")
print((REPORTS_DIR / "prueba_abierta.txt").read_text(encoding="utf-8").strip())

Primera apertura: 2026-11-05 18:09:33 · propósito: evaluación única de la etapa 7 (decisiones congeladas el 2026-11-05 17:42 (UTC−5))
Nueva lectura: 2026-11-05 18:17:27 · propósito: comprobar que models/model.joblib es el modelo evaluado (no decide nada)


In [7]:
from tarifalatam.models.price import to_usd_price

sugerido = to_usd_price(exportado.predict(test[COLUMNAS]), factor)
mae_exportado = float(np.mean(np.abs(sugerido - test["price_usd"].to_numpy())))
mae_etapa7 = etapa7["prueba"]["modelo"]["mae"]
assert abs(mae_exportado - mae_etapa7) < 1e-9, "el modelo exportado no es el que se evaluó"
print(f"MAE con models/model.joblib: {mae_exportado:.6f} USD · etapa 7: {mae_etapa7:.6f} USD → es el mismo modelo")

MAE con models/model.joblib: 41.399173 USD · etapa 7: 41.399173 USD → es el mismo modelo


## 5. La ficha del modelo

`models/ficha_modelo.md` se completa **a mano** con las cifras de las etapas 7 y 8 (no con cifras que no se midieron).

In [8]:
texto = (ROOT / "models" / "ficha_modelo.md").read_text(encoding="utf-8")
pendientes = texto.count("_______________")
print(f"ficha_modelo.md: {pendientes} campos en blanco" if pendientes else "ficha_modelo.md completa")

ficha_modelo.md completa


## 6. Siguiente: la app

Con `models/model.joblib` y `models/input_schema.json` en su sitio:

```bash
streamlit run app/streamlit_app.py
```

Comprobar: el formulario aparece, una predicción normal responde con precio y rango, **«No sé» no rompe nada**, Belice muestra
su aviso. El despliegue en internet es la clase 5 (pieza 004).